## Тестирование производительность MPS и CUDA

In [1]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import time

print('Python version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
print('MPS available:', torch.backends.mps.is_available())

Python version: 3.13.11 (main, Dec  5 2025, 16:06:33) [Clang 17.0.0 (clang-1700.4.4.1)]
PyTorch version: 2.12.0
Torchvision version: 0.27.0
CUDA available: False
MPS available: True


### Создание датасета MNIST

In [ ]:
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    transform=transforms.ToTensor(),
    download=True
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    transform=transforms.ToTensor(),
    download=True
)

def collate_fn(batch):
    x, y = zip(*batch)
    x = torch.stack(x)
    y = torch.tensor(y)
    y = F.one_hot(y, num_classes=10).float()
    return x, y

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, collate_fn=collate_fn)

### Архитектура сети

In [3]:
class MyNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.Matrix1 = nn.Linear(28**2, 100)
        self.Matrix2 = nn.Linear(100, 50)
        self.Matrix3 = nn.Linear(50, 10)
        self.R = nn.ReLU()
    def forward(self, x):
        x = x.view(-1, 28**2)
        x = self.R(self.Matrix1(x))
        x = self.R(self.Matrix2(x))
        x = self.Matrix3(x)
        return x

### Обучение

In [4]:
def train_model(model, train_loader, test_loader, epochs=5, lr=0.01, device_name="cpu"):
    device = torch.device(device_name)
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    L = nn.CrossEntropyLoss()

    history = []

    for epoch in range(1, epochs + 1):
        start_time = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            if y.ndim == 2:
                y = y.argmax(dim=1)

            optimizer.zero_grad()
            out = model(x)
            loss = L(out, y)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * x.size(0)
            train_correct += (out.argmax(dim=1) == y).sum().item()
            train_total += x.size(0)

        model.eval()
        test_loss_sum = 0.0
        test_correct = 0
        test_total = 0

        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                y = y.to(device)

                if y.ndim == 2:
                    y = y.argmax(dim=1)

                out = model(x)
                loss = L(out, y)

                test_loss_sum += loss.item() * x.size(0)
                test_correct += (out.argmax(dim=1) == y).sum().item()
                test_total += x.size(0)

        epoch_time = time.perf_counter() - start_time

        row = {
            "epoch": epoch,
            "device": device_name,
            "train_loss": train_loss_sum / train_total,
            "train_acc": train_correct / train_total,
            "test_loss": test_loss_sum / test_total,
            "test_acc": test_correct / test_total,
            "epoch_time_s": epoch_time,
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"device={device_name} | "
            f"time={epoch_time:.2f}s | "
            f"train_loss={row['train_loss']:.4f} | "
            f"train_acc={row['train_acc']:.4f} | "
            f"test_loss={row['test_loss']:.4f} | "
            f"test_acc={row['test_acc']:.4f}"
        )

    return model, history

In [6]:
f = MyNeuralNetwork()
f, history = train_model(
    model=f,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=5,
    lr=0.01,
    device_name="mps"   # или "cpu", или "cuda"
)

Epoch 01/5 | device=mps | time=21.13s | train_loss=0.4500 | train_acc=0.8650 | test_loss=0.2020 | test_acc=0.9375
Epoch 02/5 | device=mps | time=21.64s | train_loss=0.1731 | train_acc=0.9487 | test_loss=0.1302 | test_acc=0.9601
Epoch 03/5 | device=mps | time=21.27s | train_loss=0.1195 | train_acc=0.9644 | test_loss=0.1118 | test_acc=0.9671
Epoch 04/5 | device=mps | time=21.26s | train_loss=0.0927 | train_acc=0.9718 | test_loss=0.1021 | test_acc=0.9687
Epoch 05/5 | device=mps | time=20.92s | train_loss=0.0751 | train_acc=0.9765 | test_loss=0.0866 | test_acc=0.9752
